<a href="https://colab.research.google.com/github/skrixh/enterprise-retail-data-platform/blob/main/python/26MAS1015/ERDP_Data_Cleansing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import json
import xml.etree.ElementTree as ET
import os

Load all datasets

In [3]:
# POS
pos = pd.read_csv("/content/pos_transactions.csv")

# E-Commerce
ecommerce = pd.read_excel("/content/ecommerce_orders.xlsx")

# CRM
crm = pd.read_json("/content/customers.json")

# Supplier staging file
supplier = pd.read_csv("/content/stg_suppliers_orders.txt")

# Inventory XML
tree = ET.parse("/content/inventory.xml")
root = tree.getroot()

inventory_data = []

for item in root.findall("item"):
    record = {child.tag: child.text for child in item}
    inventory_data.append(record)

inventory = pd.DataFrame(inventory_data)

print("POS:", pos.shape)
print("E-Commerce:", ecommerce.shape)
print("CRM:", crm.shape)
print("Inventory:", inventory.shape)
print("Supplier:", supplier.shape)

POS: (10000, 8)
E-Commerce: (5000, 7)
CRM: (323, 3)
Inventory: (2015, 5)
Supplier: (2000, 8)


Clean POS

In [4]:
# Make a copy
pos_clean = pos.copy()

# Standardize column names
pos_clean.columns = (
    pos_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Remove exact duplicate rows
pos_clean = pos_clean.drop_duplicates()

# Convert data types
pos_clean["quantity"] = pd.to_numeric(
    pos_clean["quantity"], errors="coerce"
)

pos_clean["unitprice"] = pd.to_numeric(
    pos_clean["unitprice"], errors="coerce"
)

pos_clean["invoicedate"] = pd.to_datetime(
    pos_clean["invoicedate"], errors="coerce"
)

# Standardize text
pos_clean["description"] = pos_clean["description"].astype("string").str.strip()
pos_clean["country"] = pos_clean["country"].astype("string").str.strip()

# Handle missing description
pos_clean["description"] = pos_clean["description"].fillna("Unknown Product")

# Keep missing customer IDs as UNKNOWN
pos_clean["customerid"] = pos_clean["customerid"].astype("string")
pos_clean["customerid"] = pos_clean["customerid"].fillna("UNKNOWN")

print("Original rows:", len(pos))
print("Clean rows:", len(pos_clean))
print("Duplicates remaining:", pos_clean.duplicated().sum())
print("Missing values:")
print(pos_clean.isnull().sum())

Original rows: 10000
Clean rows: 9804
Duplicates remaining: 0
Missing values:
invoiceno      0
stockcode      0
description    0
quantity       0
invoicedate    0
unitprice      0
customerid     0
country        0
dtype: int64


Clean E-Commerce

In [5]:
ecommerce_clean = ecommerce.copy()

# Standardize column names
ecommerce_clean.columns = (
    ecommerce_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Remove duplicates
ecommerce_clean = ecommerce_clean.drop_duplicates()

# Convert numeric fields
ecommerce_clean["quantity"] = pd.to_numeric(
    ecommerce_clean["quantity"], errors="coerce"
)

ecommerce_clean["unit_price"] = pd.to_numeric(
    ecommerce_clean["unit_price"], errors="coerce"
)

# Convert date
ecommerce_clean["order_date"] = pd.to_datetime(
    ecommerce_clean["order_date"], errors="coerce"
)

# Standardize text
ecommerce_clean["country"] = (
    ecommerce_clean["country"]
    .astype("string")
    .str.strip()
)

# Handle missing customer IDs
ecommerce_clean["customer_id"] = (
    ecommerce_clean["customer_id"]
    .astype("string")
    .fillna("UNKNOWN")
)

print("Original rows:", len(ecommerce))
print("Clean rows:", len(ecommerce_clean))
print("Duplicates remaining:", ecommerce_clean.duplicated().sum())
print("\nMissing values:")
print(ecommerce_clean.isnull().sum())

Original rows: 5000
Clean rows: 4951
Duplicates remaining: 0

Missing values:
order_id       0
customer_id    0
product_id     0
quantity       0
unit_price     0
order_date     0
country        0
dtype: int64


Clean CRM

In [6]:
crm_clean = crm.copy()

# Standardize column names
crm_clean.columns = (
    crm_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Remove duplicates
crm_clean = crm_clean.drop_duplicates()

# Clean customer ID
crm_clean["customer_id"] = (
    crm_clean["customer_id"]
    .astype("string")
    .str.strip()
)

# Clean country
crm_clean["country"] = (
    crm_clean["country"]
    .astype("string")
    .str.strip()
)

print("Original rows:", len(crm))
print("Clean rows:", len(crm_clean))
print("Duplicates remaining:", crm_clean.duplicated().sum())
print("\nMissing values:")
print(crm_clean.isnull().sum())

Original rows: 323
Clean rows: 323
Duplicates remaining: 0

Missing values:
customer_id    0
country        0
source_note    0
dtype: int64


Clean Inventory

In [7]:
inventory_clean = inventory.copy()

# Standardize column names
inventory_clean.columns = (
    inventory_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Remove duplicates
inventory_clean = inventory_clean.drop_duplicates()

# Convert numeric fields
inventory_clean["unit_price"] = pd.to_numeric(
    inventory_clean["unit_price"], errors="coerce"
)

inventory_clean["stock_quantity"] = pd.to_numeric(
    inventory_clean["stock_quantity"], errors="coerce"
)

inventory_clean["reorder_level"] = pd.to_numeric(
    inventory_clean["reorder_level"], errors="coerce"
)

# Clean text
inventory_clean["product_id"] = (
    inventory_clean["product_id"]
    .astype("string")
    .str.strip()
)

inventory_clean["product_description"] = (
    inventory_clean["product_description"]
    .astype("string")
    .str.strip()
)

# Handle missing descriptions
inventory_clean["product_description"] = (
    inventory_clean["product_description"]
    .fillna("Unknown Product")
)

print("Original rows:", len(inventory))
print("Clean rows:", len(inventory_clean))
print("Duplicates remaining:", inventory_clean.duplicated().sum())

print("\nMissing values:")
print(inventory_clean.isnull().sum())

Original rows: 2015
Clean rows: 2015
Duplicates remaining: 0

Missing values:
product_id             0
product_description    0
unit_price             0
stock_quantity         0
reorder_level          0
dtype: int64


Clean Supplier

In [8]:
supplier_clean = supplier.copy()

# Standardize column names
supplier_clean.columns = (
    supplier_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Remove duplicates
supplier_clean = supplier_clean.drop_duplicates()

# Convert numeric fields
supplier_clean["quantity"] = pd.to_numeric(
    supplier_clean["quantity"], errors="coerce"
)

supplier_clean["unit_cost"] = pd.to_numeric(
    supplier_clean["unit_cost"], errors="coerce"
)

# Convert dates
supplier_clean["order_date"] = pd.to_datetime(
    supplier_clean["order_date"], errors="coerce"
)

supplier_clean["expected_delivery_date"] = pd.to_datetime(
    supplier_clean["expected_delivery_date"], errors="coerce"
)

# Clean text columns
text_columns = [
    "purchase_order_id",
    "supplier_id",
    "product_id",
    "order_status"
]

for col in text_columns:
    supplier_clean[col] = (
        supplier_clean[col]
        .astype("string")
        .str.strip()
    )

print("Original rows:", len(supplier))
print("Clean rows:", len(supplier_clean))
print("Duplicates remaining:", supplier_clean.duplicated().sum())

print("\nMissing values:")
print(supplier_clean.isnull().sum())

Original rows: 2000
Clean rows: 2000
Duplicates remaining: 0

Missing values:
purchase_order_id         0
supplier_id               0
product_id                0
quantity                  0
unit_cost                 0
order_date                0
expected_delivery_date    0
order_status              0
dtype: int64


Create the cleaned datasets

In [12]:
os.makedirs("/content/cleaned_data", exist_ok=True)

pos_clean.to_csv(
    "/content/cleaned_data/pos_transactions_clean.csv",
    index=False
)

ecommerce_clean.to_csv(
    "/content/cleaned_data/ecommerce_orders_clean.csv",
    index=False
)

crm_clean.to_csv(
    "/content/cleaned_data/customers_clean.csv",
    index=False
)

inventory_clean.to_csv(
    "/content/cleaned_data/inventory_clean.csv",
    index=False
)

supplier_clean.to_csv(
    "/content/cleaned_data/supplier_orders_clean.csv",
    index=False
)

print("All cleaned datasets created successfully!")

All cleaned datasets created successfully!


Final quality check

In [11]:
datasets = {
    "POS": pos_clean,
    "E-Commerce": ecommerce_clean,
    "CRM": crm_clean,
    "Inventory": inventory_clean,
    "Supplier": supplier_clean
}

for name, df in datasets.items():

    print("\n" + "=" * 50)
    print(name)

    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Duplicates:", df.duplicated().sum())
    print("Missing values:", df.isnull().sum().sum())


POS
Rows: 9804
Columns: 8
Duplicates: 0
Missing values: 0

E-Commerce
Rows: 4951
Columns: 7
Duplicates: 0
Missing values: 0

CRM
Rows: 323
Columns: 3
Duplicates: 0
Missing values: 0

Inventory
Rows: 2015
Columns: 5
Duplicates: 0
Missing values: 0

Supplier
Rows: 2000
Columns: 8
Duplicates: 0
Missing values: 0
